# Reference copy - needs non-public data

이 노트북은 구독 플랫폼 개발 중 쓴 **포트폴리오 계산 프로토타입의 참고용 사본**입니다. 입력인 `data.parquet`(종목별 일간 수익률, 전략 점수, 시가총액 분위; 운영 데이터)은 포함하지 않았으므로 **그대로는 실행되지 않습니다.** 출력과 로컬 경로는 제거했습니다. 같은 로직을 합성 데이터로 실행하려면 `../code/portfolio_engine.py`를 사용하세요.

In [ ]:
import pandas as pd

# parquet 파일 로드
df = pd.read_parquet("data.parquet")

In [ ]:
# 3. 포트폴리오 일별 수익률 계산 (최적화 버전)
def calculate_daily_portfolio_returns_optimized(df, rebalance_dates, strategy_col, n_stocks, position, size_decile_min=None, size_decile_max=None):
    """
    포트폴리오 일별 수익률 계산 (성능 최적화)
    
    주요 개선사항:
    1. pivot 연산을 한 번만 수행 (반복문 밖으로 이동)
    2. 불필요한 copy() 제거
    3. 벡터화된 연산 활용
    """
    # 전체 데이터에 대해 pivot을 미리 수행 (가장 비용이 큰 연산)
    daily_ret_pivot = df.pivot_table(index='Date', columns='Ticker', values='daily_ret', aggfunc='first')
    
    all_daily_returns = []
    
    for i in range(len(rebalance_dates)):
        rebal_date = rebalance_dates[i]
        
        # 다음 리밸런싱 날짜
        if i < len(rebalance_dates) - 1:
            next_rebal_date = rebalance_dates[i + 1]
        else:
            next_rebal_date = df['Date'].max()
        
        # 리밸런싱일에 종목 선정 (copy 제거)
        rebal_data = df[df['Date'] == rebal_date]
        
        # Size Decile 범위 필터링
        if size_decile_min is not None and size_decile_max is not None and 'Size Decile' in rebal_data.columns:
            rebal_data = rebal_data[(rebal_data['Size Decile'] >= size_decile_min) & 
                                   (rebal_data['Size Decile'] <= size_decile_max)]
        
        # 결측치 제거
        rebal_data = rebal_data.dropna(subset=[strategy_col])
        
        if len(rebal_data) == 0:
            continue
        
        # Position에 따른 종목 선정
        if position == 1:
            selected_tickers = rebal_data.nlargest(n_stocks, strategy_col)['Ticker'].tolist()
        elif position == -1:
            selected_tickers = rebal_data.nsmallest(n_stocks, strategy_col)['Ticker'].tolist()
        elif position == 0:
            long_tickers = rebal_data.nlargest(n_stocks, strategy_col)['Ticker'].tolist()
            short_tickers = rebal_data.nsmallest(n_stocks, strategy_col)['Ticker'].tolist()
        
        # 보유 기간 인덱스 슬라이싱 (boolean indexing보다 빠름)
        period_data = daily_ret_pivot.loc[(daily_ret_pivot.index >= rebal_date) & 
                                          (daily_ret_pivot.index <= next_rebal_date)]
        
        if len(period_data) == 0:
            continue
        
        # Position에 따른 일별 수익률 계산
        if position == 0:  # Long-Short
            long_pivot = period_data[long_tickers].fillna(0)
            short_pivot = period_data[short_tickers].fillna(0)
            
            long_cumulative = ((long_pivot + 1).cumprod().mean(axis=1)).pct_change().dropna()
            short_cumulative = ((short_pivot + 1).cumprod().mean(axis=1)).pct_change().dropna()
            
            period_returns = long_cumulative - short_cumulative
        else:
            if position == 1:
                selected_pivot = period_data[selected_tickers].fillna(0)
                period_returns = ((selected_pivot + 1).cumprod().mean(axis=1)).pct_change().dropna()
            else:  # position == -1
                selected_pivot = period_data[selected_tickers].fillna(0)
                period_returns = -((selected_pivot + 1).cumprod().mean(axis=1)).pct_change().dropna()
        
        all_daily_returns.append(period_returns)
    
    # 모든 기간의 일별 수익률 합치기
    if len(all_daily_returns) > 0:
        daily_returns = pd.concat(all_daily_returns).sort_index()
        daily_returns.name = 'daily_return'
        return daily_returns
    else:
        return pd.Series(name='daily_return')
    
# 리밸런싱 날짜 결정 로직
def get_rebalance_dates(df, rebalance_day):
    """
    각 월별로 리밸런싱 날짜를 찾는 함수
    rebalance_day에 가장 가까운 날짜를 선택 (해당 월에 그 날짜가 없으면 월말)
    """
    # Date를 datetime으로 변환 (필요시)
    df['Date'] = pd.to_datetime(df['Date'])

    # 각 월의 리밸런싱 기준일 찾기
    df['year_month'] = df['Date'].dt.to_period('M')
    df['day'] = df['Date'].dt.day

    rebalance_dates = []
    
    for ym in df['year_month'].unique():
        month_data = df[df['year_month'] == ym]
        
        # 해당 월에서 rebalance_day 이상인 날짜 중 가장 작은 날짜 선택
        # (예: 31일 기준이면 31일이 있으면 31일, 없으면 그 달의 마지막 날)
        month_dates = month_data['Date'].unique()
        month_days = month_data['day'].unique()
        
        # rebalance_day와 같거나 큰 날짜 중 가장 작은 것
        valid_days = month_days[month_days >= rebalance_day]
        if len(valid_days) > 0:
            selected_day = valid_days.min()
        else:
            # rebalance_day보다 작으면 그 달의 마지막 날
            selected_day = month_days.max()
        
        selected_date = month_data[month_data['day'] == selected_day]['Date'].iloc[0]
        rebalance_dates.append(selected_date)
    
    return sorted(rebalance_dates)

In [ ]:
# 전략 파라미터 설정
n_stocks = 20  # 종목 수
strategy = "jackpot"  # 전략 이름
size_decile_min = 1  # 시가총액 분위 수
size_decile_max = 1  # 시가총액 분위 수
rebalance_day = 31  # 리밸런싱 기준일
position = 1  # 포지션

In [ ]:
rebalance_dates = get_rebalance_dates(df, rebalance_day)

portfolio_daily = calculate_daily_portfolio_returns_optimized(
    df, 
    rebalance_dates, 
    strategy, 
    n_stocks, 
    position,
    size_decile_min=size_decile_min,
    size_decile_max=size_decile_max
)

In [ ]:
portfolio_daily
